# 4단계: 음식 임베딩 실험

라벨링된 5,219개 음식 메뉴(라벨링단위 기준)를 자연어 임베딩으로 변환하고 자연어 검색 품질을 확인한다.

검증된 로직은 모두 `src/embedding/`에 있다. 이 노트북은 그 함수들을 실제 데이터에 대해 실행하고 결과를 확인하는 용도이며, 로직을 다시 구현하지 않는다.

- 모델/텍스트 설계 근거: `docs/embedding/MODEL_SELECTION.md`, `docs/embedding/TEXT_DESIGN.md`
- 입력: `data/processed/labeling/labeling_units.csv`, `labels_chat_full_v3.jsonl`, `data/processed/food_menu.csv`
- 출력: `data/embeddings/<모델_리비전_텍스트구성_설정해시>/` (벡터 + manifest)

기존에 저장된 384차원 벡터(`data/embeddings/embeddings_v1_text*.npy`)는 이전 모델(e5-small) 결과이며 이 노트북에서 건드리지 않는다.

## 1. 모듈 로드 및 실행 환경 확인

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import (
    DEFAULT_SPEC, E5Embedder, EmbeddingStore,
    build_config, build_records, choose_batch_size, compute_input_hash,
    cosine_top_k, describe_environment, detect_device,
    join_units_labels, load_joined, load_sources,
    prepare_variant, run_variant, search, source_hashes, validate_vectors,
)

print("모듈 로드 완료")

모듈 로드 완료


In [2]:
env = describe_environment()
for k, v in env.items():
    print(f"{k}: {v}")

doc_batch_size = choose_batch_size(env["selected_device"], env["total_memory_gb"])
print(f"\n선택 배치 크기: {doc_batch_size}")

platform: macOS-26.5.2-arm64-arm-64bit
processor: arm
cpu_count: 8
total_memory_gb: 8.0
torch_version: 2.14.0
cuda_available: False
mps_available: True
selected_device: mps

선택 배치 크기: 32


## 2. 모델 규격

`intfloat/multilingual-e5-base`, 리비전 고정, 접두어·pooling·정규화는 모델 카드 규격을 그대로 따른다 (근거: `docs/embedding/MODEL_SELECTION.md`).

In [3]:
pd.Series(DEFAULT_SPEC.as_dict())

model_id                      intfloat/multilingual-e5-base
revision           d128750597153bb5987e10b1c3493a34e5a4502a
dimension                                               768
query_prefix                                        query: 
document_prefix                                   passage: 
pooling                                                mean
normalize                                              True
max_seq_length                                          512
license                                                 mit
dtype: object

## 3. 데이터 연결 및 검증

라벨링단위ID 기준으로 라벨(`labels_chat_full_v3.jsonl`), 라벨링 단위(`labeling_units.csv`), 음식 데이터(`food_menu.csv`)를 연결하고 불일치를 검증한다.

In [4]:
joined, report = load_joined()
print(report.summary())
print(f"\n연결 상태 깨끗함(is_clean): {report.is_clean}")
assert report.is_clean, "데이터 연결에 불일치가 있습니다"
assert len(joined) == 5219, f"기대한 5,219건과 다릅니다: {len(joined)}"

라벨링 단위 5219건, 라벨 5219건, 음식 8032행
연결 성공 5219건
ID 중복: 단위 0건, 라벨 0건
라벨 없는 단위 0건, 단위 없는 라벨 0건
식품코드목록 불일치 0건
음식 데이터에 없는 식품코드 0건
라벨링 대상 외 식품코드 0건

연결 상태 깨끗함(is_clean): True


## 4. 텍스트 구성 비교

- A: 메뉴명 + 대표식품명 + 식품대분류명
- B: A + 매운맛, 국물, 제공온도, 조리법, 기름짐, 든든함 (미확인 제외)
- 업체명, 사이즈, 라벨링단위ID, 근거 문장, 라벨출처 등 관리 정보는 텍스트에서 제외한다 (근거: `docs/embedding/TEXT_DESIGN.md`)

In [5]:
sample = joined[:10]
texts_a, records_a = build_records(sample, "A")
texts_b, records_b = build_records(sample, "B")

for i, (a, b) in enumerate(zip(texts_a, texts_b), 1):
    print(f"[{i}] A: {a}")
    print(f"    B: {b}")

[1] A: 치폴레쉬림프디트로이트 피자 빵 및 과자류
    B: 치폴레쉬림프디트로이트 피자 빵 및 과자류 매운맛 약함 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[2] A: 소고기스튜 찜류
    B: 소고기스튜 찜류 매운맛 없음 국물약간 제공온도 뜨거움 조리법 끓임 기름짐 보통 든든함 보통
[3] A: 칠리 핫도그 핫도그 빵 및 과자류
    B: 칠리 핫도그 핫도그 빵 및 과자류 매운맛 약함 국물없음 제공온도 따뜻함 조리법 혼합 기름짐 보통 든든함 보통
[4] A: 슈퍼 디럭스 히어로 피자 더블치즈 페퍼로니 엣지 피자 빵 및 과자류
    B: 슈퍼 디럭스 히어로 피자 더블치즈 페퍼로니 엣지 피자 빵 및 과자류 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[5] A: 콤비네이션 피자 피자 빵 및 과자류
    B: 콤비네이션 피자 피자 빵 및 과자류 매운맛 없음 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[6] A: 7번가스페셜 피자 씬도우 피자 빵 및 과자류
    B: 7번가스페셜 피자 씬도우 피자 빵 및 과자류 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[7] A: 닭튀김 살코기 닭튀김 튀김류
    B: 닭튀김 살코기 닭튀김 튀김류 매운맛 없음 국물없음 제공온도 뜨거움 조리법 튀김 기름짐 높음
[8] A: 페파로니 피자 씬 피자 빵 및 과자류
    B: 페파로니 피자 씬 피자 빵 및 과자류 매운맛 없음 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[9] A: 콤비네이션바이트 피자 피자 빵 및 과자류
    B: 콤비네이션바이트 피자 피자 빵 및 과자류 매운맛 없음 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음
[10] A: 페페로니 러버 피자 오리지널 파마산 치즈 피자 빵 및 과자류
    B: 페페로니 러버 피자 오리지널 파마산 치즈 피자 빵 및 과자류 매운맛 없음 국물없음 제공온도 뜨거움 조리법 오븐 기름짐 높음


## 5. 작은 샘플로 실행·저장·로드 확인

전체 5,219개를 실행하기 전에 20개 샘플로 임베딩 생성, 저장, 재로드가 정상 동작하는지 확인한다. 임시 디렉터리에 저장하므로 `data/embeddings/`에는 영향을 주지 않는다.

In [6]:
import tempfile, time

t0 = time.time()
embedder = E5Embedder(spec=DEFAULT_SPEC, batch_size=doc_batch_size)
print(f"모델 로드 시간: {time.time() - t0:.1f}초, device={embedder.device}, batch_size={embedder.batch_size}")

with tempfile.TemporaryDirectory() as tmp:
    sample_store = EmbeddingStore(tmp)
    t0 = time.time()
    sample_vectors, sample_manifest, sample_info = run_variant(
        "A", joined[:20], embedder=embedder, store=sample_store,
    )
    print(f"샘플 20개 인코딩+저장 시간: {time.time() - t0:.1f}초")
    print("형태:", sample_vectors.shape, sample_vectors.dtype)

    reloaded, reloaded_manifest = sample_store.load(sample_info["name"])
    assert reloaded.shape == sample_vectors.shape
    assert len(reloaded_manifest["records"]) == 20
    print("저장/재로드 검증 통과")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

모델 로드 시간: 9.9초, device=mps, batch_size=32


/Users/imjeonghyeog/workspace/Menu_recommend/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

샘플 20개 인코딩+저장 시간: 1.1초
형태: (20, 768) float32
저장/재로드 검증 통과


## 6. 전체 임베딩 생성 (A, B)

5,219건 전체에 대해 텍스트 A, B 임베딩을 생성한다. 같은 설정으로 이미 저장된 결과가 있으면 재사용하고, 없으면 새로 계산해 `data/embeddings/<모델_리비전_텍스트구성_설정해시>/`에 저장한다.

In [7]:
store = EmbeddingStore()
sources = source_hashes()

t0 = time.time()
vectors_a, manifest_a, info_a = run_variant(
    "A", joined, embedder=embedder, store=store, sources=sources,
)
print(f"텍스트 A: {vectors_a.shape}, 재사용={info_a['reused']} ({info_a['reason']}), 소요 {time.time()-t0:.1f}초")
print("저장 위치:", store.result_dir(info_a["name"]))

텍스트 A: (5219, 768), 재사용=True (설정 일치), 소요 0.1초
저장 위치: /Users/imjeonghyeog/workspace/Menu_recommend/data/embeddings/multilingual-e5-base_d1287505_textA_v1_eb332ded


In [8]:
t0 = time.time()
vectors_b, manifest_b, info_b = run_variant(
    "B", joined, embedder=embedder, store=store, sources=sources,
)
print(f"텍스트 B: {vectors_b.shape}, 재사용={info_b['reused']} ({info_b['reason']}), 소요 {time.time()-t0:.1f}초")
print("저장 위치:", store.result_dir(info_b["name"]))

텍스트 B: (5219, 768), 재사용=True (설정 일치), 소요 0.1초
저장 위치: /Users/imjeonghyeog/workspace/Menu_recommend/data/embeddings/multilingual-e5-base_d1287505_textB_v1_f87e895f


In [9]:
stats_a = validate_vectors(vectors_a, DEFAULT_SPEC.dimension, DEFAULT_SPEC.normalize)
stats_b = validate_vectors(vectors_b, DEFAULT_SPEC.dimension, DEFAULT_SPEC.normalize)
print("텍스트 A 검증:", stats_a)
print("텍스트 B 검증:", stats_b)
assert stats_a["num_vectors"] == stats_b["num_vectors"] == 5219

텍스트 A 검증: {'num_vectors': 5219, 'dimension': 768, 'dtype': 'float32', 'finite': True, 'min_norm': 0.9999998807907104, 'max_norm': 1.0000001192092896}
텍스트 B 검증: {'num_vectors': 5219, 'dimension': 768, 'dtype': 'float32', 'finite': True, 'min_norm': 0.9999998807907104, 'max_norm': 1.0000001192092896}


## 7. 자연어 검색 실험

사용자 문장을 같은 모델로 `query: ` 접두어를 붙여 임베딩하고, 코사인 유사도로 Top-5를 찾는다.

In [10]:
test_queries = [
    "비 오는 날 얼큰한 국물 먹고 싶어",
    "맵지 않고 따뜻한 음식",
    "차갑고 가볍게 먹을 메뉴",
    "바삭하고 기름진 음식",
    "든든한 밥 한 끼",
    "국물 없는 매운 음식",
    "상큼하고 시원한 음식",
    "포만감 있는 저녁밥",
    "빠르게 먹을 수 있는 간식",
    "따뜻한 국이나 찌개",
    "느끼하지 않은 담백한 음식",
    "단짠단짠한 음식",
]

query_vectors = embedder.encode_queries(test_queries, show_progress=False)
print(f"{len(test_queries)}개 질의 임베딩 완료: {query_vectors.shape}")

12개 질의 임베딩 완료: (12, 768)


In [11]:
rows_a = search(query_vectors, vectors_a, manifest_a["records"], test_queries, k=5)
rows_b = search(query_vectors, vectors_b, manifest_b["records"], test_queries, k=5)

df_a = pd.DataFrame(rows_a)
df_a.insert(0, "텍스트구성", "A")
df_b = pd.DataFrame(rows_b)
df_b.insert(0, "텍스트구성", "B")

results_df = pd.concat([df_a, df_b], ignore_index=True)
results_df

,텍스트구성,질의,순위,메뉴명,업체명,유사도,주요라벨,라벨링단위ID,대표식품명,식품대분류명
0,A,비 오는 날 얼큰한 국물 먹고 싶어,1,얼갈이 된장국,-,0.8277,"매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움",54e1d2c4dc40,얼갈이 된장국,국 및 탕류
1,A,비 오는 날 얼큰한 국물 먹고 싶어,2,냉이 된장국,-,0.8262,"매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움",f740f5fee27a,냉이 된장국,국 및 탕류
2,A,비 오는 날 얼큰한 국물 먹고 싶어,3,비빔국수,-,0.8262,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",7976df5ba7df,비빔국수,면 및 만두류
3,A,비 오는 날 얼큰한 국물 먹고 싶어,4,국수,-,0.8243,"조리법 끓임, 기름짐 낮음, 든든함 보통",a3f30c8201e1,국수,면 및 만두류
4,A,비 오는 날 얼큰한 국물 먹고 싶어,5,올갱이국수,-,0.8231,"매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통",18b0ea2badf9,올갱이국수,면 및 만두류
...,...,...,...,...,...,...,...,...,...,...
115,B,단짠단짠한 음식,1,단짠반반 피자,서오릉피자,0.8611,"국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음",10661271e5e8,피자,빵 및 과자류
116,B,단짠단짠한 음식,2,단짠반반,서오릉피자,0.8592,"국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음",522bc8c28712,피자,빵 및 과자류
117,B,단짠단짠한 음식,3,단짠콘후라이 피자,피자는치즈빨,0.8516,"매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음",42072b3839b8,피자,빵 및 과자류
118,B,단짠단짠한 음식,4,불닭바베큐 피자 씬도우,피자파는집,0.8439,"매운맛 강함, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통",ddaf1b1b2c4e,피자,빵 및 과자류


In [12]:
out_path = PROJECT_ROOT / "data" / "embeddings" / "search_results.csv"
results_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"저장됨: {out_path}")

저장됨: /Users/imjeonghyeog/workspace/Menu_recommend/data/embeddings/search_results.csv


In [13]:
for query in test_queries:
    print(f"\n[{query}]")
    print("  A:", " / ".join(f"{r['메뉴명']}({r['유사도']})" for r in rows_a if r["질의"] == query))
    print("  B:", " / ".join(f"{r['메뉴명']}({r['유사도']})" for r in rows_b if r["질의"] == query))


[비 오는 날 얼큰한 국물 먹고 싶어]
  A: 얼갈이 된장국(0.8277) / 냉이 된장국(0.8262) / 비빔국수(0.8262) / 국수(0.8243) / 올갱이국수(0.8231)
  B: 올갱이해장국(0.8354) / 뼈다귀해장국(0.8337) / 올갱이국(0.829) / 국수 김치말이국수(0.8288) / 올갱이국수(0.8282)

[맵지 않고 따뜻한 음식]
  A: 볶음 우동(0.8383) / 무 된장국(0.8382) / 짬뽕(0.8381) / 달콤딥고구마밀도우(0.8378) / 불고기치즈 바게트(0.8361)
  B: 전여친 생각 토스트(0.8481) / 햄&치즈 프레시 샌드위치(0.8454) / 꽃맛살쉬림프(0.845) / 게살듬뿍모닝 샌드위치(0.8449) / 그릴치킨&당근라페샌드위치(0.8444)

[차갑고 가볍게 먹을 메뉴]
  A: 간편조리세트 차돌박이숙주볶음(0.8639) / 스위트불고기(0.8627) / 고추장불고기(0.8625) / 소불고기 피자(0.8622) / 스위트불고기 피자(0.8622)
  B: 간편조리세트 차돌박이숙주볶음(0.8687) / 간편조리세트 대파고추장불고기(0.8619) / 할라불고기 피자(0.8613) / 간편조리세트 소고기야채말이(0.8608) / 간편조리세트 매콤 콩나물불고기(0.8599)

[바삭하고 기름진 음식]
  A: 쫄면(0.8515) / 라면(0.847) / LA BBQ 불고기치즈크러스트 골드(0.8462) / 바삭몬테크리스토(0.8458) / 치즈크러스트흑마늘불고기피자(0.8456)
  B: 바삭옥수수통새우피자(0.8489) / 바삭옥수수통새우피자 치즈크러스트(0.8471) / 바삭옥수수통새우피자 리치골드크러스트(0.8464) / 불고기흑미치즈크러스트(0.8439) / 킹 브레드 쉬림프 골드 칠리 피자 크림치즈(0.8436)

[든든한 밥 한 끼]
  A: 달콤딥고구마밀도우(0.8293) / 다넣었어 피자(0.8274) / 불닭바베큐밀도우(0.8271) / 불고기 1인피자(0.8264) / 달

## 8. A/B 결과 관찰

아래 지표는 실제 검색 결과를 세는 관찰용 수치이며, 어느 쪽이 더 정확하다는 근거로 쓰지 않는다.

In [14]:
top1_a = {q: next(r for r in rows_a if r["질의"] == q and r["순위"] == 1)["라벨링단위ID"] for q in test_queries}
top1_b = {q: next(r for r in rows_b if r["질의"] == q and r["순위"] == 1)["라벨링단위ID"] for q in test_queries}
same_top1 = [q for q in test_queries if top1_a[q] == top1_b[q]]

print(f"A/B 1순위가 같은 질의: {len(same_top1)}/{len(test_queries)}건")
for q in test_queries:
    mark = "동일" if top1_a[q] == top1_b[q] else "다름"
    print(f"  [{mark}] {q}")

A/B 1순위가 같은 질의: 1/12건
  [다름] 비 오는 날 얼큰한 국물 먹고 싶어
  [다름] 맵지 않고 따뜻한 음식
  [동일] 차갑고 가볍게 먹을 메뉴
  [다름] 바삭하고 기름진 음식
  [다름] 든든한 밥 한 끼
  [다름] 국물 없는 매운 음식
  [다름] 상큼하고 시원한 음식
  [다름] 포만감 있는 저녁밥
  [다름] 빠르게 먹을 수 있는 간식
  [다름] 따뜻한 국이나 찌개
  [다름] 느끼하지 않은 담백한 음식
  [다름] 단짠단짠한 음식


In [15]:
# 부정 조건 관찰: "맵지 않고 따뜻한 음식"에서 실제로 매운 메뉴가 상위에 나오는지
neg_query = "맵지 않고 따뜻한 음식"
uid_to_spicy = {j["라벨링단위ID"]: j["label"]["라벨"]["매운맛"] for j in joined}

for variant, rows in (("A", rows_a), ("B", rows_b)):
    top5 = [r for r in rows if r["질의"] == neg_query]
    spicy_levels = [uid_to_spicy.get(r["라벨링단위ID"], "미확인") for r in top5]
    print(f"{variant}: {[(r['메뉴명'], s) for r, s in zip(top5, spicy_levels)]}")

A: [('볶음 우동', '없음'), ('무 된장국', '없음'), ('짬뽕', '강함'), ('달콤딥고구마밀도우', '없음'), ('불고기치즈 바게트', '없음')]
B: [('전여친 생각 토스트', '미확인'), ('햄&치즈 프레시 샌드위치', '없음'), ('꽃맛살쉬림프', '없음'), ('게살듬뿍모닝 샌드위치', '없음'), ('그릴치킨&당근라페샌드위치', '없음')]


In [16]:
# 피자 편중 관찰: 전체 검색 결과 top-5 중 피자 비중
for variant, rows in (("A", rows_a), ("B", rows_b)):
    total = len(rows)
    pizza = sum(1 for r in rows if "피자" in r["메뉴명"] or "피자" in r["대표식품명"])
    print(f"{variant}: 전체 {total}건 중 피자 포함 {pizza}건 ({pizza * 100 // total}%)")

A: 전체 60건 중 피자 포함 28건 (46%)
B: 전체 60건 중 피자 포함 19건 (31%)


In [17]:
# 동일 메뉴 변형 반복 관찰: 한 질의의 top-5 안에서 대표식품명이 겹치는 경우
for variant, rows in (("A", rows_a), ("B", rows_b)):
    repeated_queries = []
    for q in test_queries:
        names = [r["대표식품명"] for r in rows if r["질의"] == q]
        if len(set(names)) < len(names):
            repeated_queries.append(q)
    print(f"{variant}: top-5 내 대표식품명 중복이 있는 질의 {len(repeated_queries)}/{len(test_queries)}건 -> {repeated_queries}")

A: top-5 내 대표식품명 중복이 있는 질의 7/12건 -> ['차갑고 가볍게 먹을 메뉴', '바삭하고 기름진 음식', '든든한 밥 한 끼', '상큼하고 시원한 음식', '포만감 있는 저녁밥', '느끼하지 않은 담백한 음식', '단짠단짠한 음식']
B: top-5 내 대표식품명 중복이 있는 질의 8/12건 -> ['맵지 않고 따뜻한 음식', '차갑고 가볍게 먹을 메뉴', '바삭하고 기름진 음식', '상큼하고 시원한 음식', '빠르게 먹을 수 있는 간식', '따뜻한 국이나 찌개', '느끼하지 않은 담백한 음식', '단짠단짠한 음식']


## 9. 결과 요약

In [18]:
print("모델:", DEFAULT_SPEC.model_id)
print("리비전:", DEFAULT_SPEC.revision)
print("실행 장치:", embedder.device, "배치 크기:", embedder.batch_size)
print()
print("텍스트 A:", vectors_a.shape, "->", store.result_dir(info_a["name"]))
print("텍스트 B:", vectors_b.shape, "->", store.result_dir(info_b["name"]))
print()
print("저장된 임베딩 결과 목록:")
for row in store.list_results():
    print(" ", row)

모델: intfloat/multilingual-e5-base
리비전: d128750597153bb5987e10b1c3493a34e5a4502a
실행 장치: mps 배치 크기: 32

텍스트 A: (5219, 768) -> /Users/imjeonghyeog/workspace/Menu_recommend/data/embeddings/multilingual-e5-base_d1287505_textA_v1_eb332ded
텍스트 B: (5219, 768) -> /Users/imjeonghyeog/workspace/Menu_recommend/data/embeddings/multilingual-e5-base_d1287505_textB_v1_f87e895f

저장된 임베딩 결과 목록:


 {'name': 'embeddings_v1_textA_n5219.npy', 'model_id': None, 'model_revision': None, 'text_variant': None, 'num_vectors': 5219, 'dimension': 384, 'created_at': None}
  {'name': 'embeddings_v1_textB_n5219.npy', 'model_id': None, 'model_revision': None, 'text_variant': None, 'num_vectors': 5219, 'dimension': 384, 'created_at': None}
  {'name': 'multilingual-e5-base_d1287505_textA_v1_eb332ded', 'model_id': 'intfloat/multilingual-e5-base', 'model_revision': 'd1287505', 'text_variant': 'A', 'num_vectors': 5219, 'dimension': 768, 'created_at': '2026-09-22T08:02:55+00:00'}
  {'name': 'multilingual-e5-base_d1287505_textB_v1_f87e895f', 'model_id': 'intfloat/multilingual-e5-base', 'model_revision': 'd1287505', 'text_variant': 'B', 'num_vectors': 5219, 'dimension': 768, 'created_at': '2026-09-22T08:03:45+00:00'}


## 5단계로 넘길 점

위 8절의 관찰(부정 조건, 피자 편중, 메뉴 반복)을 바탕으로 실제로 확인된 한계만 적는다. 코사인 유사도만으로는 "맵지 않고"처럼 텍스트에 없는 부정 조건을 걸러낼 수 없고, 상위 후보에 특정 대분류가 몰리거나 같은 메뉴의 변형이 함께 올라오는 현상은 위 셀의 실제 출력으로 확인한다. 5단계에서는 이 관찰을 근거로 후보 확장(Top-K를 넓게 뽑기), 속성 기반 하드 필터, 재랭킹 도입 여부를 결정한다. 이번 4단계에서는 재랭킹이나 필터를 구현하지 않았다.